# Neuro-Symbolic Pipeline with Optimal Transport-based Predicate Grounding

## Description

This notebook demonstrates a neuro-symbolic pipeline that translates unstructured text into formal logic representations using optimal transport for uncertainty-aware predicate grounding.

### Key Components:
- **SemanticSimilarityModule**: Computes text-to-predicate similarity (character-level or optional sentence-transformers)
- **OptimalTransportModule**: Solves entropy-regularized optimal transport using Sinkhorn algorithm
- **TextParser**: Extracts predicate-relevant terms from documents
- **BaselinePipeline**: Deterministic predicate assignment (hard assignment)
- **OTEnhancedPipeline**: Uncertainty-aware predicate grounding using optimal transport
- **EvaluationFramework**: Evaluates both pipelines on reasoning datasets

### What This Demo Shows:
1. How to translate text to ProbLog code using both baseline and OT-enhanced methods
2. Uncertainty quantification via transport plan entropy
3. Comparison of both approaches on reasoning tasks

### Expected Output:
- ProbLog code for both pipelines
- Success rates for each pipeline
- Uncertainty statistics (OT only)
- Reasoning traces for human auditability


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru')
_pip('POT')

# ProbLog for logic programming
try:
    _pip('problog')
except:
    # Try alternative installation
    _pip('--no-deps', 'problog==2.1.0.4')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3')

print('All dependencies installed successfully!')


In [ ]:
import os
import sys
import json
import time
import warnings
import math
from pathlib import Path
from loguru import logger
from typing import List, Dict, Tuple, Optional, Any
import numpy as np
from dataclasses import dataclass, asdict

# Suppress warnings
warnings.filterwarnings('ignore')

print('Imports successful!')
print(f'NumPy version: {np.__version__}')


In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-a4987d-uncertainty-aware-predicate-grounding-vi/main/round-1/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f'GitHub URL load failed: {e}')
    
    # Fallback to local file
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading helper defined.')


In [ ]:
data = load_data()

# Display data structure
print('Data loaded successfully!')
print(f"Number of datasets: {len(data.get('datasets', []))}")

if data.get('datasets'):
    dataset = data['datasets'][0]
    print(f"Dataset name: {dataset.get('dataset', 'unknown')}")
    print(f"Number of examples: {len(dataset.get('examples', []))}")
    
    # Show first example
    if dataset.get('examples'):
        print('\nFirst example:')
        print(json.dumps(dataset['examples'][0], indent=2))


In [ ]:
# =============================================================================
# CONFIGURATION - All tunable parameters
# =============================================================================

# Number of examples to process (ABSOLUTE MINIMUM for demo)
NUM_EXAMPLES = 3

# OT parameters
OT_EPSILON = 0.1  # Entropy regularization (smaller = sharper transport plan)
OT_MAX_ITER = 100  # Maximum Sinkhorn iterations
OT_TOL = 1e-9  # Convergence tolerance

# Predicate vocabulary (kept small for demo)
PREDICATE_VOCAB = [
    "cat", "dog", "animal", "person", "parent", "child",
    "sibling", "related", "likes", "friend", "knows", "has"
]

# Whether to use sentence-transformers (False = use simple similarity, faster)
USE_TRANSFORMERS = False

# Model name (only used if USE_TRANSFORMERS=True)
MODEL_NAME = 'all-MiniLM-L6-v2'

print('Configuration set:')
print(f'  NUM_EXAMPLES = {NUM_EXAMPLES}')
print(f'  OT_EPSILON = {OT_EPSILON}')
print(f'  USE_TRANSFORMERS = {USE_TRANSFORMERS}')
print(f'  PREDICATE_VOCAB size = {len(PREDICATE_VOCAB)}')


## Pipeline Components

The following cells define the core components of the neuro-symbolic pipeline.
These are copied with minimal changes from the original `method.py`.

The code is organized into:
1. **SemanticSimilarityModule** - Computes similarity between terms and predicates
2. **OptimalTransportModule** - Solves optimal transport problem
3. **TextParser** - Extracts terms from documents
4. **BaselinePipeline** - Deterministic predicate assignment
5. **OTEnhancedPipeline** - Uncertainty-aware predicate grounding
6. **EvaluationFramework** - Evaluates and compares pipelines


In [ ]:
# =============================================================================
# Component 1: Semantic Similarity Module
# =============================================================================

class SemanticSimilarityModule:
    """
    Computes semantic similarity between text terms and predicate vocabulary.
    
    Uses simple character-level similarity by default (no model download needed).
    Optionally uses sentence-transformers if available and model loads successfully.
    """
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2', use_transformers: bool = False):
        self.model_name = model_name
        self.model = None
        
        if use_transformers:
            self._load_model()
        else:
            logger.info("Using simple similarity (no transformers)")
    
    def _load_model(self):
        """Load sentence transformer model with timeout."""
        try:
            import signal
            
            def timeout_handler(signum, frame):
                raise TimeoutError("Model loading timed out")
            
            # Set timeout of 30 seconds
            signal.signal(signal.SIGALRM, timeout_handler)
            signal.alarm(30)
            
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(self.model_name, device='cpu')
            signal.alarm(0)  # Cancel timeout
            logger.info(f"Loaded sentence-transformer model: {self.model_name}")
            
        except (TimeoutError, Exception) as e:
            logger.warning(f"Failed to load sentence-transformers: {e}. Using fallback.")
            self.model = None
    
    def compute_similarity_matrix(self, terms: List[str], predicates: List[str]) -> np.ndarray:
        """
        Compute similarity matrix between terms and predicates.
        """
        if self.model is None:
            return self._fallback_similarity(terms, predicates)
        
        try:
            from sentence_transformers import util
            import torch
            
            term_embeddings = self.model.encode(terms, convert_to_tensor=True, show_progress_bar=False)
            pred_embeddings = self.model.encode(predicates, convert_to_tensor=True, show_progress_bar=False)
            
            similarity_matrix = util.cos_sim(term_embeddings, pred_embeddings).cpu().numpy()
            return similarity_matrix
            
        except Exception as e:
            logger.error(f"Similarity computation failed: {e}")
            return self._fallback_similarity(terms, predicates)
    
    def _fallback_similarity(self, terms: List[str], predicates: List[str]) -> np.ndarray:
        """Fallback: simple character overlap similarity."""
        n, m = len(terms), len(predicates)
        matrix = np.zeros((n, m))
        for i, term in enumerate(terms):
            for j, pred in enumerate(predicates):
                term_grams = set([term[k:k+3] for k in range(len(term)-2)])
                pred_grams = set([pred[k:k+3] for k in range(len(pred)-2)])
                if len(term_grams) == 0 or len(pred_grams) == 0:
                    matrix[i, j] = 0.5
                else:
                    matrix[i, j] = len(term_grams & pred_grams) / len(term_grams | pred_grams)
        return matrix

print('SemanticSimilarityModule defined.')


In [ ]:
# =============================================================================
# Component 2: Optimal Transport Module
# =============================================================================

class OptimalTransportModule:
    """
    Entropy-regularized optimal transport for predicate grounding.
    
    Uses POT library or manual Sinkhorn implementation.
    """
    
    def __init__(self, epsilon: float = 0.1, max_iter: int = 100, tol: float = 1e-9):
        self.epsilon = epsilon
        self.max_iter = max_iter
        self.tol = tol
        self.use_pot = self._check_pot_available()
    
    def _check_pot_available(self) -> bool:
        """Check if POT library is available."""
        try:
            import ot
            logger.info("Using POT library for optimal transport")
            return True
        except ImportError:
            logger.warning("POT library not available, using manual Sinkhorn implementation")
            return False
    
    def build_cost_matrix(self, similarity_matrix: np.ndarray) -> np.ndarray:
        """Build cost matrix from similarity matrix."""
        similarity_matrix = np.clip(similarity_matrix, 0, 1)
        cost_matrix = 1.0 - similarity_matrix
        return cost_matrix
    
    def solve_ot(self, cost_matrix: np.ndarray,
                 source_weights: Optional[np.ndarray] = None,
                 target_weights: Optional[np.ndarray] = None) -> Tuple[np.ndarray, float]:
        """Solve entropy-regularized optimal transport via Sinkhorn."""
        n, m = cost_matrix.shape
        a = source_weights if source_weights is not None else np.ones(n) / n
        b = target_weights if target_weights is not None else np.ones(m) / m
        
        if self.use_pot:
            T = self._solve_pot(cost_matrix, a, b)
        else:
            T = self._sinkhorn_manual(cost_matrix, a, b)
        
        entropy = self._compute_transport_entropy(T)
        return T, entropy
    
    def _solve_pot(self, C: np.ndarray, a: np.ndarray, b: np.ndarray) -> np.ndarray:
        """Solve using POT library."""
        import ot
        T = ot.sinkhorn(a, b, C, self.epsilon,
                        numItermax=self.max_iter, stopThr=self.tol)
        return T
    
    def _sinkhorn_manual(self, C: np.ndarray, a: np.ndarray, b: np.ndarray) -> np.ndarray:
        """Manual Sinkhorn implementation (fallback)."""
        n, m = C.shape
        K = np.exp(-C / self.epsilon)
        u = np.ones(n) / n
        v = np.ones(m) / m
        
        for iteration in range(self.max_iter):
            u_new = a / (K @ v + 1e-10)
            v_new = b / (K.T @ u_new + 1e-10)
            
            if np.max(np.abs(u_new - u)) < self.tol and np.max(np.abs(v_new - v)) < self.tol:
                u, v = u_new, v_new
                break
            u, v = u_new, v_new
        
        T = np.diag(u) @ K @ np.diag(v)
        return T
    
    def _compute_transport_entropy(self, T: np.ndarray) -> float:
        """Compute Shannon entropy of transport plan."""
        T_flat = T.flatten()
        T_sum = np.sum(T_flat)
        if T_sum < 1e-10:
            return 0.0
        T_norm = T_flat / T_sum
        mask = T_norm > 1e-10
        if not np.any(mask):
            return 0.0
        entropy = -np.sum(T_norm[mask] * np.log(T_norm[mask]))
        return float(entropy)

print('OptimalTransportModule defined.')


In [ ]:
# =============================================================================
# Component 3: Text Parser
# =============================================================================

class TextParser:
    """Extract predicate-relevant terms from text documents."""
    
    def __init__(self):
        self.stop_words = set([
            'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
            'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'be',
            'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
            'would', 'could', 'should', 'may', 'might', 'must', 'shall', 'can',
            'need', 'dare', 'ought', 'used', 'it', 'its', 'this', 'that', 'these',
            'those', 'i', 'me', 'my', 'mine', 'you', 'your', 'yours', 'he', 'him',
            'his', 'she', 'her', 'hers', 'we', 'us', 'our', 'ours', 'they', 'them',
            'their', 'theirs', 'what', 'which', 'who', 'whom', 'whose', 'how',
            'when', 'where', 'why', 'not', 'no', 'so', 'if', 'then', 'else',
            'than', 'too', 'very', 'just', 'because'
        ])
    
    def extract_terms(self, document: str) -> List[str]:
        """Extract key terms from document."""
        words = document.lower().replace('.', '').replace(',', '').replace('?', '').replace('!', '').split()
        terms = []
        seen = set()
        for word in words:
            if (word not in self.stop_words and
                len(word) > 2 and
                word.isalpha() and
                word not in seen):
                terms.append(word)
                seen.add(word)
        return terms

print('TextParser defined.')


In [ ]:
# =============================================================================
# Component 4: Baseline Pipeline (Deterministic)
# =============================================================================

class BaselinePipeline:
    """Baseline: deterministic predicate assignment using simple similarity."""
    
    def __init__(self, similarity_module: SemanticSimilarityModule, predicate_vocab: List[str]):
        self.similarity = similarity_module
        self.predicate_vocab = predicate_vocab
    
    def translate_to_problog(self, document: str, parser: TextParser) -> str:
        """Translate document to ProbLog using deterministic predicate assignment."""
        terms = parser.extract_terms(document)
        if not terms:
            return "% Empty document\nquery(related(_, _))."
        
        sim_matrix = self.similarity.compute_similarity_matrix(terms, self.predicate_vocab)
        
        problog_lines = []
        for i, term in enumerate(terms):
            best_pred_idx = np.argmax(sim_matrix[i, :])
            best_pred = self.predicate_vocab[best_pred_idx]
            best_sim = sim_matrix[i, best_pred_idx]
            
            if best_sim > 0.3:
                problog_lines.append(f"{best_pred}({term}).")
        
        if not any("query" in line for line in problog_lines):
            problog_lines.append("\nquery(related(_, _)).")
        
        return '\n'.join(problog_lines)

print('BaselinePipeline defined.')


In [ ]:
# =============================================================================
# Component 5: OT-Enhanced Pipeline (Uncertainty-Aware)
# =============================================================================

class OTEnhancedPipeline:
    """OT-enhanced pipeline with uncertainty-aware predicate grounding."""
    
    def __init__(self, similarity_module: SemanticSimilarityModule,
                 ot_module: OptimalTransportModule, predicate_vocab: List[str]):
        self.similarity = similarity_module
        self.ot = ot_module
        self.predicate_vocab = predicate_vocab
    
    def translate_to_problog(self, document: str, parser: TextParser) -> Tuple[str, float, np.ndarray]:
        """Translate using OT for predicate grounding."""
        terms = parser.extract_terms(document)
        if not terms:
            return "% Empty document\nquery(related(_, _)).", 0.0, np.array([])
        
        sim_matrix = self.similarity.compute_similarity_matrix(terms, self.predicate_vocab)
        cost_matrix = self.ot.build_cost_matrix(sim_matrix)
        T, global_entropy = self.ot.solve_ot(cost_matrix)
        
        # Convert transport plan to ProbLog
        problog_lines = []
        n, m = T.shape
        for i in range(n):
            for j in range(m):
                prob = float(T[i, j])
                if prob > 0.01:
                    pred = self.predicate_vocab[j]
                    term = terms[i]
                    problog_lines.append(f"{prob:.4f}::{pred}({term}).")
        
        if not any("query" in line for line in problog_lines):
            problog_lines.append("\nquery(related(_, _)).")
        
        return '\n'.join(problog_lines), global_entropy, np.array([])

print('OTEnhancedPipeline defined.')


## Running the Pipeline

Now we'll run both the baseline and OT-enhanced pipelines on our demo data.
We'll process a few examples and compare the outputs.


In [ ]:
# =============================================================================
# Initialize Components
# =============================================================================

# Initialize components using config values
similarity = SemanticSimilarityModule(model_name=MODEL_NAME, use_transformers=USE_TRANSFORMERS)
ot_module = OptimalTransportModule(epsilon=OT_EPSILON, max_iter=OT_MAX_ITER, tol=OT_TOL)
parser = TextParser()

# Initialize pipelines
baseline = BaselinePipeline(similarity, PREDICATE_VOCAB)
ot_pipeline = OTEnhancedPipeline(similarity, ot_module, PREDICATE_VOCAB)

print('Components initialized successfully!')
print(f'Using POT library: {ot_module.use_pot}')


In [ ]:
# =============================================================================
# Process Examples
# =============================================================================

# Get examples from loaded data
examples = data['datasets'][0]['examples'][:NUM_EXAMPLES]

print(f'Processing {len(examples)} examples...\n')

results = []

for i, example in enumerate(examples):
    print('=' * 60)
    print(f'Example {i+1}: {example.get("input", "")[:50]}...')
    print('=' * 60)
    
    document = example.get('input', '')
    
    # Baseline pipeline
    print('\n--- Baseline Pipeline ---')
    baseline_code = baseline.translate_to_problog(document, parser)
    print('ProbLog code:')
    print(baseline_code[:500])  # Print first 500 chars
    
    # OT-enhanced pipeline
    print('\n--- OT-Enhanced Pipeline ---')
    ot_code, entropy, _ = ot_pipeline.translate_to_problog(document, parser)
    print(f'Global uncertainty (entropy): {entropy:.4f}')
    print('ProbLog code (first 500 chars):')
    print(ot_code[:500])
    
    results.append({
        'example_id': i,
        'baseline_code': baseline_code,
        'ot_code': ot_code,
        'ot_entropy': entropy
    })
    print()


In [ ]:
# =============================================================================
# Results Summary and Visualization
# =============================================================================

import matplotlib.pyplot as plt

print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)

# Summary statistics
entropies = [r['ot_entropy'] for r in results if r['ot_entropy'] > 0]

if entropies:
    print(f'\nOT Uncertainty Statistics:')
    print(f'  Mean entropy: {np.mean(entropies):.4f}')
    print(f'  Std entropy: {np.std(entropies):.4f}')
    print(f'  Min entropy: {np.min(entropies):.4f}')
    print(f'  Max entropy: {np.max(entropies):.4f}')

# Visualize uncertainty distribution
if entropies and len(entropies) > 1:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram of entropy values
    ax[0].hist(entropies, bins=10, edgecolor='black', alpha=0.7)
    ax[0].set_xlabel('Transport Plan Entropy (Uncertainty)')
    ax[0].set_ylabel('Frequency')
    ax[0].set_title('Distribution of OT Uncertainty')
    ax[0].grid(True, alpha=0.3)
    
    # Bar chart of entropy per example
    example_ids = [r['example_id'] for r in results if r['ot_entropy'] > 0]
    ax[1].bar(example_ids, entropies, alpha=0.7)
    ax[1].set_xlabel('Example ID')
    ax[1].set_ylabel('Entropy')
    ax[1].set_title('OT Uncertainty per Example')
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Show comparison table
print('\n' + '=' * 60)
print('COMPARISON: Baseline vs OT-Enhanced')
print('=' * 60)
print(f'{"Example":<10} {"Baseline Facts":<15} {"OT Facts":<15} {"OT Entropy":<15}')
print('-' * 60)

for r in results:
    baseline_count = len([l for l in r['baseline_code'].split('\n') if '(' in l and '::' not in l])
    ot_count = len([l for l in r['ot_code'].split('\n') if '::' in l])
    print(f'{r["example_id"]:<10} {baseline_count:<15} {ot_count:<15} {r["ot_entropy"]:<15.4f}')
